# 03. ML 전달 데이터 확인
최종 데이터 상태와 universe 정합성을 한 번에 확인합니다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from src.validate import validate_ml_dataset
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
final_df = pd.read_parquet(PROCESSED_DIR / 'final_df.parquet')
coverage_df = pd.read_csv(PROCESSED_DIR / 'coverage_df.csv', parse_dates=['actual_start_date', 'actual_end_date'])
final_missing_df = pd.read_csv(PROCESSED_DIR / 'final_missing_df.csv')
sp500_universe = pd.read_csv(RAW_DIR / 'sp500_universe.csv')

In [ ]:
summary = {
 'final_df shape': final_df.shape, '최종 종목 수': final_df['Ticker'].nunique(),
 '전체 행 수': len(final_df), '날짜 범위': (final_df['Date'].min(), final_df['Date'].max()),
 '컬럼 목록': final_df.columns.tolist(), 'final_missing_df 종목 수': final_missing_df['Ticker'].nunique(),
 'short_history 종목 수': int(coverage_df['short_history'].sum()),
 'Ticker/Date 중복': int(final_df.duplicated(['Ticker','Date']).sum()),
 '필수 컬럼 결측': final_df[['Ticker','Date','Open','High','Low','Close','Volume']].isna().sum().to_dict()}
for key, value in summary.items(): print(f'{key}: {value}')
display(coverage_df.describe(include='all').T)
display(final_missing_df.groupby('fail_stage').size().rename('count'))

In [ ]:
report = validate_ml_dataset(final_df, final_missing_df, sp500_universe, coverage_df)
print('[통과] 필수 검증 통과')
report